# Group 42

Main

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

from thymio import Thymio
from vision import Vision

import motion_controll
import filtering
import local_nav
import global_nav
import utils

In [ ]:
thymio = Thymio(pos_init=[0, 0], orient=0)
await thymio._connect_to_thymio_()
thymio.stop()

Local nav tests

In [ ]:
thymio.nav_mode = "LOCAL"
while True: 
    await thymio.update_ir()
    # is_object = local_nav.is_object(thymio)
    # if(is_object):
    #     print("object detected")
    print(str(thymio.ir_sensors) + "                     ", end="\r")
    # avoid_right = local_nav.avoid_right(thymio, grid)
    local_nav.avoid_obstacle(thymio, 0, 0, True)
# await thymio.update_ir()
# print(thymio.ir_sensors)

Motion controll testing

In [ ]:
thymio.pos = [20,20]
thymio.orient = 0
goal = [22,25]
print(motion_controll.follow_path(thymio, goal))

In [ ]:
await thymio.unlock()

## Main Loop

In [ ]:
thymio.stop()

In [ ]:
# External modules import
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

# Internal modules and thymio class import
from thymio import Thymio
from vision import Vision
import motion_controll
import filtering
import local_nav
import global_nav
import utils

# Constants
GL_NAV_CHANGE_THLD = 20  # Number of cycles without obstacle to switch back to global navigation
WAIT_AFTER_KIDNAP = 20

# Initalisation of the grid
visionInstance = Vision() # Calls getEnvironment which stores the arena and creates the grid
# visionInstance.display_grid()

# Get cell size for coordinate conversions
cell_size_cm = visionInstance.getCellSizeCm()

# Get start position and orientation
ret, frame = visionInstance.cap.read() #Taking a single image to find the start pos of the robot
if not ret:
    raise ValueError("Camera failed to capture the frame.")

#start_pos, start_orient = visionInstance.getRobotPose(frame)
start_pos, start_orient = visionInstance.getInitialRobotPose()
print(start_pos, start_orient)

# Convert to grid coordinates (start_pos is in meters, convert to cm first)
if start_pos:
    start_cell = utils.real_to_grid((start_pos[0]*100, start_pos[1]*100))
    # print(f"start: {start_cell}")
    goal_pos = visionInstance.getGoalPos()
    goal_cell = utils.real_to_grid((goal_pos[0]*100, goal_pos[1]*100))  # Goal in cm
    # print(f"goal: {goal_cell}")
    # visionInstance.display_grid(start_cell, goal_cell)

# Get the grid from vision
grid = visionInstance.grid

# Find path to goal (mode = "a_star" or "djikstra")
# 0 for djikstra, 1 for a*
path_find_mode = 1
path, expanded_grid = global_nav.find_path(path_find_mode, grid, start_cell, goal_cell)
global_nav.display_grid(expanded_grid, start_cell, goal_cell)

# Initialisation of the thymio and connexion
thymio = Thymio(pos_init=start_pos, orient=start_orient)
await thymio._connect_to_thymio_()
thymio.stop()

# Local nav variables
gl_nav_change_cntr = 0
kidnaped_cntr = 0
avoid_right = True
LN_orient = 0
LN_pos_at_obst = [0, 0]
LN_stage = 0
obstacle_avoided = False
is_kidnaped = False

# filter initialisation
x_est, P_est, Q, R = filtering.init_filter(utils.q_x, utils.q_y, utils.q_theta, utils.q_v, utils.r_x,
                                           utils.r_y, start_pos*100, start_orient) #carefull maybe start_pos in meters

# Initialisation of robot position data and global path for visualisations
cam_robot_positions_cm = []
filter_robot_positions_cm = []
global_path_points_cm = path

while(True):
    print(f"Nav_mode: {thymio.nav_mode}")
    start_loop_time = time.time()
    await thymio.update_ir()
    await thymio.update_ground_sensors()

    # print(thymio.ir_sensors)
    is_object = local_nav.is_object(thymio)
    is_kidnaped = local_nav.check_kidnap(thymio)

    if(is_kidnaped):
        thymio.nav_mode = "KIDNAPPED"
        thymio.stop()

    # is_object = False
    if(is_object and thymio.nav_mode=="GLOBAL"):
        avoid_right = local_nav.avoid_right(thymio, grid)
        thymio.nav_mode = "LOCAL"
    
    if(obstacle_avoided):
        if(thymio.nav_mode=="GLOBAL"):
            print("Error: obstacle_avoided true in global nav")
            break
        else:
            obstacle_avoided = False
            LN_stage = 0
            thymio.nav_mode = "GLOBAL"
            robot_cell = utils.real_to_grid((thymio.pos[0], thymio.pos[1]))
            path, expanded_grid = global_nav.find_path(path_find_mode, grid, robot_cell, goal_cell)
    
    if(thymio.nav_mode == "KIDNAPPED" and not is_kidnaped):
        kidnaped_cntr += 1
        if kidnaped_cntr < WAIT_AFTER_KIDNAP:
            ret, frame = visionInstance.cap.read() 
            if not ret:
                print("Camera failed to capture the frame.")
                break

            vis = frame.copy()
            result = visionInstance.getRobotPoseAndVisualise(frame, vis)

            # Check if detection failed (returns (None, None))
            if result[0] is None:
                print("Robot not detected during kidnap wait")
            else:
                print("Robot detected after kidnap wait")  
                print("Results[0] from vision: ", result[0]) 
                kidnaped_cntr = 0
                thymio.nav_mode = "GLOBAL"
                thymio.pos[0] = result[0][0]*100
                thymio.pos[1] = result[0][1]*100
                thymio.orient = result[1]
                print("Robot pos after kidnap: ", thymio.pos, " orient: ", thymio.orient)
                robot_cell = utils.real_to_grid((thymio.pos[0], thymio.pos[1]))
                path, expanded_grid = global_nav.find_path(path_find_mode, grid, robot_cell, goal_cell)
                print("Robot re-localized, switching to GLOBAL navigation")

    match thymio.nav_mode:
        case "KIDNAPPED":
            # Stop the robot
            thymio.stop()
        
        case "LOCAL":
            # local navigation to avoid obstacle
            obstacle_avoided, LN_stage, LN_pos_at_obst, LN_orient = local_nav.avoid_obstacle(thymio, avoid_right, LN_stage, LN_pos_at_obst, LN_orient)

        case "GLOBAL":
            next_wp = path[0]
            # next_wp = [35.0, 35.0]  # Copy to avoid modifying the original
            # print(f"thymio pos: {thymio.pos}, next wp: {next_wp}", end="\r")
            wp_reached = motion_controll.follow_path(thymio, next_wp)
            if(wp_reached):
                # print("Waypoint reached!")
                path.pop(0)  # Supprime le waypoint atteint
                if len(path) == 0:  # Si plus de waypoints
                    print("Goal reached")
                    thymio.stop()
                    break  # Sortir de la boucle

    
    # stores the last orientation
    thymio.last_orient = thymio.orient
    
    # pos_on_img, orient_on_img = vision.get_pos()
    # Get robot position from vision 
    ret, frame = visionInstance.cap.read() 
    if not ret:
        print("Camera failed to capture the frame.")
        break

    vis = frame.copy() # Copying frame so we can display shapes on top without affecting detection
    visionInstance.visualiseArena(vis, visionInstance.arena_corners_pixels)
    # visualise global nav path
    visionInstance.visualiseGlobalPath(vis, global_path_points_cm)

    # --- Always detect and draw robot pose ---
    result = visionInstance.getRobotPoseAndVisualise(frame, vis)
    
    # Check if detection failed (returns (None, None))
    if result[0] is None:
        x_est, P_est = filtering.filter_pos(thymio, [None, None, None, 0], x_est, # *100 to pu it in cm
                                        P_est, Q, R, 1/utils.FREQ_MAIN_LOOP, utils.RATIO_SPEED)
        # print("Robot not detected only filtering")
    else:
        [X_robot, Y_robot], robot_heading_angle = result
        # print(f"X: {X_robot:.5f}, Y: {Y_robot:.5f}, Direction: {robot_heading_angle:.5f}", end="\r")
        x_est, P_est = filtering.filter_pos(thymio, [X_robot*100, Y_robot*100, robot_heading_angle, 0], x_est, # *100 to pu it in cm
                                        P_est, Q, R, 1/utils.FREQ_MAIN_LOOP, utils.RATIO_SPEED)
        # print(f"Robot pos from cam :{X_robot*100:.2f} cm, {Y_robot*100:.2f} cm, orient: {robot_heading_angle:.2f} rad")
    
    
    # print(f"Robot pos from filter :{x_est[0]:.2f} cm, {x_est[1]:.2f} cm, orient: {x_est[2]:.2f} rad")


    thymio.pos = [x_est[0], x_est[1]]  # already in cm
    thymio.orient = x_est[2]

    # Visualise before and after filtering
    if (result[0] is not None):
        cam_robot_positions_cm.append([result[0][0]*100, result[0][1]*100]) # Convert m to cm
    filter_robot_positions_cm.append([x_est[0], x_est[1]])

    visionInstance.visualiseGlobalPath(vis, filter_robot_positions_cm, (0, 215, 255))
    visionInstance.visualiseGlobalPath(vis, cam_robot_positions_cm, (255, 0, 0))

    cv2.imshow("Live camera", vis)
    if(cv2.waitKey(1) & 0xFF == ord('q')):
        break


    # filtering.filter_pos(thymio, pos_on_img, orient_on_img)
    time_main_loop = time.time() - start_loop_time
    if(time_main_loop < 1/utils.FREQ_MAIN_LOOP):
        # print(f"Wait for : {1/utils.FREQ_MAIN_LOOP - time_main_loop}")
        await asyncio.sleep(1/utils.FREQ_MAIN_LOOP - time_main_loop)
    # print(f"main loop time : {time_main_loop}")

cv2.destroyAllWindows()

In [ ]:
thymio.stop()

In [ ]:

# Get robot pose in loop
if not visionInstance.cap.isOpened(): # Checking access to camera feed
    print("Error: Could not access the webcam.")
    exit()

while True:
    ret, frame = visionInstance.cap.read() # Taking an image frame from the camera feed
    if not ret:
        print("Camera failed to capture the frame.")
        break

    vis = frame.copy() # Copying frame so we can display shapes on top without affecting detection

    # --- Always redraw arena outline ---
    visionInstance.visualiseArena(vis, visionInstance.arena_corners_pixels)

    # --- Always detect and draw robot pose ---
    [X, Y], robot_heading_angle = visionInstance.getRobotPoseAndVisualise(frame, vis)
    if (X is None or Y is None or robot_heading_angle is None):
        continue
    print(f"X: {X:.5f}, Y: {Y:.5f}, Direction: {robot_heading_angle:.5f}")

    # --- Show the live window ---
    cv2.imshow("Live Robot Pose", vis)

    # Exit on Q
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

visionInstance.cap.release()
cv2.destroyAllWindows()

In [ ]:
a = [[1,2],[3,4]]
a[1][1]

In [ ]:
await thymio.update_ir()
print(thymio.ir_sensors[0:5])

In [ ]:
thymio.set_motor_speeds([100, 100])
while True:
    await thymio.update_ir()
    if sum(thymio.ir_sensors) > 2000:
        thymio.stop()
        break
    

Next cell makes the Thymio robot move forward for 4 seconds and then stops each time the Forward button is pressed. Program stops when the Backward button is pressed.  
It is intended to collect data for computing the **velocity variance**.

In [ ]:
await thymio.button_loop()

Using this program to measure (with a ruler) the distance travelled by the bot each time to see differencies despite constant time and compute the variance on speed state.

In [ ]:
import numpy as np
data_velocity_error=[142, 138, 138, 137, 139, 137, 138, 135, 142, 141] #distances in mm travelled at presumed same speed for a constant time
data_velocity_error=[x/4 for x in data_velocity_error] #distances divided by the constant time to get true velocities

mean_speed=np.mean(data_velocity_error) #mean speed in mm/s
print(mean_speed)
ratio_speed=100/mean_speed #from tests above, for a speed of 100 we get a mean speed of 34.675 mm/s

q_v = np.var(data_velocity_error) # variance on speed state
print(q_v)

<img src="position_measurement.png" width="400">

From the camera we got a data set of XY position measurements from the same position to search for some differencies and compute **variances on XY states and measurements**.

In [ ]:
measurements_from_camera=np.array([[13.118, 13.303], [13.153, 13.317], [13.090, 13.307], [13.062, 13.255], [13.059, 13.281],
                                   [13.074, 13.321], [13.026, 13.273], [13.073, 13.295], [13.023, 13.270]])
x_measurements = [x[0] for x in measurements_from_camera]
y_measurements = [y[1] for y in measurements_from_camera]

var_x = np.var(x_measurements)
var_y = np.var(y_measurements)

q_x = var_x/2 #variance on x position state
r_x = var_x/2 #variance on x position measurement
q_y = var_y/2 #variance on y position state
r_y = var_y/2 #variance on y position measurement
print(q_x, q_y, r_x, r_y)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import asyncio

from thymio import Thymio
from vision import Vision

import motion_controll
import filtering
import local_nav
import global_nav
import utils

thymio = Thymio(pos_init=[0, 0], orient=0)
await thymio._connect_to_thymio_()
thymio.stop()

visionInstance = Vision()
visionInstance.display_grid()

ret, frame = visionInstance.cap.read()
print(ret)
start_theta= visionInstance.getInitialRobotPose()[1]
print(start_theta)
if start_theta is None:
    raise RuntimeError("Robot not detected in initial frame")

thymio.set_motor_speeds([0,0])
delta_t = 2
k=10
data_theta_pos=np.zeros(k)
data_theta_neg=np.zeros(k)

for i in range(k):

    thymio.set_motor_speeds([-100,100])
    start_time=time.time()
    while time.time() - start_time < delta_t:
        await asyncio.sleep(0.1)
    thymio.set_motor_speeds([0, 0])
    ret, frame = visionInstance.cap.read()
    vis=frame.copy()
    on_going_theta=visionInstance.getRobotPoseAndVisualise(frame, vis)[1]
    if on_going_theta is None:
        print("Robot not detected (positive turn), skipping this measurement")
        continue
    data_theta_neg[i]=on_going_theta-start_theta

    start_theta=on_going_theta
    thymio.set_motor_speeds([100,-100])
    start_time=time.time()
    while time.time() - start_time < delta_t:
        await asyncio.sleep(0.1)
    thymio.set_motor_speeds([0, 0])
    ret, frame = visionInstance.cap.read()
    vis=frame.copy()
    on_going_theta=visionInstance.getRobotPoseAndVisualise(frame, vis)[1]
    if on_going_theta is None:
        print("Robot not detected (positive turn), skipping this measurement")
        continue
    data_theta_pos[i]=on_going_theta-start_theta


print(f"theta neg : {data_theta_neg}")
print(f"theta pos : {data_theta_pos}")

In [ ]:
import numpy as np
data_theta=np.array([[1.20987582], [1.14751077], [1.14971828], [1.13941801], [1.12451291], [1.12512088],
                    [1.14892936], [1.13340664], [1.14468753], [1.14857364]])
q_theta=np.var(data_theta)
print(q_theta)

---
---

# **Group 42 Report**

---
---

# **Set up**


**Key Libraries:**  
- `numpy` - Array operations for the grid  
- `matplotlib` - Visualization of the map and path  
- `heapq` - Priority queue for efficient pathfinding  
- `cv2` (OpenCV) - Morphological operations to expand obstacles  
- `utils` - Project-specific constants and coordinate conversion functions

# **Environment**

Our environment is set up in the following way:
- white background
- aruco markers for robot detection, arena border detection and for the goal
- obstacles are cut out of red paper
We initially wanted to go with many different colours for the various detections but after testing, we realized that colours were quite difficult to detect and depended heavily on lighting, aruco markers were much more reliable.

---
---

# **Vision**

The computer vision class is responsible for identifying and visualising all features in the environment (arena, start position, goal, and global obstacles) as well the live pose of the robot. The static features of the environment are detected once at the start, whereas the pose of the robot is detected continuously throughout execution.

The class is defined in `vision.py`.

## **Key Design Choices**
* We chose to use ArUco markers for the arena corners, goal, and robot location as their high contrast provides robustness under illumination changes and the unique ID allows each feature to be individually identifiable.
* We chose to use a square arena to allow for easy conversion to a grid. The grid allows for simpler global path planning techniques (explore cells).
---

## **1. Getting the Environment**

When the Vision class is constructed, it immediately calls:

    getEnvironment()

This initialises the camera capture (stores it for later use to avoid startup delay) and extracts all static, environment-level information (arena, start, goal, obstacles) and stores it for later use. 

The function calls the following components:


### **1.1 getArenaCornerPixelsAndRealArenaSize(frame)**

##### **Purpose**

Detect two ArUco markers (IDs **1** and **2**) representing the bottom left and top right corners of the arena respectively. These markers define:
* the arena boundary in **pixel coordinates**
* the arena boundary in **world coordinates** (from printed marker size)
* the mapping between the two (homography)

##### **Theory**

ArUco detection returns the 2D corner positions of each marker.
Since we know the marker’s physical size (3cm sides), we can use the distance between corners of a marker to establish a pixel-to-meter scaling and use this to obtain the physical dimensions of the arena. We define this arena as our global coordinate system where (0,0) is at the center of the bottom left marker. We can then use the arena corners in pixel and world coordinates to define the homography for the environment.


### **1.2 getRobotPoseCameraFrame(frame)**

##### **Purpose**

Detect the ArUco marker with ID **0**, which is the marker placed on the top of the robot. 

We printed the ArUco marker so that it **covers the entire top surface**, making:

* detection more robust
* the marker centre coincide with the robot’s geometric centre
* orientation estimation accurate and consistent

##### **Theory**

OpenCV’s ArUco module provides the corner points of the robot marker ($p_i$ for $i=0,1,2,3$). These corner points can be used to identify the center coordinates, $C$:

$
C_x = \frac{1}{4} \sum_{i=0}^{3} p_{i, x}
$

$
C_y = \frac{1}{4} \sum_{i=0}^{3} p_{i, y}
$

They can also be used to calculate the robot's heading direction. In the image below you can see the top left and top right corner points of the ArUco marker with ID 0. 

<div style="text-align: center;">
  <img src="headingCalculation.png" alt="Heading calculation visualisation" style="width: 500px; height: auto;">
  <h3>Heading calculation visualisation</h3>
</div>

To get the vector that goes along the front side, you do:

$
v = v_{TR} - v_{TL}
$

To get the angle that this makes with the horizontal, you do:
$
\theta = \arctan2(-v_{\Delta y}, v_{\Delta x})
$

Note the $-\Delta y$ due to the difference between the y axis definition in the camera frame (downwards direction) and our defined frame (upwards direction). 

However, observe that this doesn't actually give you the heading direction if we view the top edge as the front of the robot, it gives you $\pi/2$ clockwise from it. So, we just placed the ArUco marker such that the right hand side represented the robot's forward direction (orange arrow).

Here is an expanded and corrected version of your Markdown section **with the full explanation added**, in a clean report-ready format.


## **1.3 cameraToGlobal (Homography)**

### **Purpose**

Convert any pixel coordinate from the camera frame into world coordinates (meters) inside the arena.

This allows us to take detections (robot pose, obstacles, goal) in image space and express them in the global coordinate frame used by navigation and planning.


### **Theory: 2D Homography**

A homography is a planar projective transformation that maps points from one plane to another.
Because the arena floor is flat, all relevant features lie on a single plane, so the homography maps:

**camera pixel space → real-world coordinate space**

A homography is represented as a 3×3 matrix:

$
H =
\begin{bmatrix}
h_{11} & h_{12} & h_{13} \\
h_{21} & h_{22} & h_{23} \\
h_{31} & h_{32} & h_{33}
\end{bmatrix}
$

Points are expressed in homogeneous coordinates, meaning the pixel (u, v) becomes:

$
\mathbf{p}_{camera} =
\begin{bmatrix}
u \\ v \\ 1
\end{bmatrix}
$

The mapping is:

$
\mathbf{p_{world}}' =
H * \mathbf{p_{camera}} =
\begin{bmatrix}
X' \\ Y' \\ W'
\end{bmatrix}
$

Since homogeneous coordinates are scale-invariant, the homography introduces a scale factor. This can be removed using normalisation. So, the real world point is recovered by dividing by (W'):

$
X = \frac{X'}{W'}
\qquad
Y = \frac{Y'}{W'}
$

### **Computing the Homography**

We obtain four pixel-space arena corners from ArUco markers:
* bottom-left
* bottom-right
* top-right
* top-left

We also define the corresponding real-world coordinates in meters:
* (0,0)
* (width, 0)
* (width, height)
* (0, height)

Given these four point correspondences, the homography (H) is solved by minimising:

$
H=argmin_H \sum_i​∥p_{world,i}​−H*p_{camera,i}​∥
$

This is a least-squares optimisation over the entries of the matrix (H).

Rather than solving this manually, we use OpenCV’s built-in solver:

`H, _ = cv2.findHomography(pixel_pts, world_pts)`

OpenCV internally performs the projective least-squares fit using the Direct Linear Transform (DLT) algorithm.

### **Using the Homography (camera → world)**

To convert any pixel point (u, v):

1. Convert to homogeneous coordinates
2. Multiply by the homography
3. Divide by (W')

$
\begin{bmatrix}
X' \\ Y' \\ W'
\end{bmatrix} = H
\begin{bmatrix}
u \\ v \\ 1
\end{bmatrix}
$

$
X = \frac{X'}{W'} \\
Y = \frac{Y'}{W'}
$

### **1.4 locateObstaclesRed(frame)**

##### **Purpose**

Detect **all red obstacles**, extract their contour boundaries, and approximate them as polygons.
Storing obstacles as polygons provides:

* shape generalisation
* an easy way to convert polygons from camera to world coordinates (using their verticies)
* a simple way to provide information to the `createGrid()` function

##### **Why red?**

After empirical testing with several colours (blue, green, yellow, black), **red** objects resulted in:
* the best contrast against the arena floor
* minimal confusion with ArUco markers (black/white)
* most stable colour segmentation across lighting conditions

##### **Theory: Colour Segmentation**

We convert the image to HSV, because hue is illumination-invariant.
Typical red thresholding uses two ranges because red wraps around the hue wheel:

```python
lower1 = (0, 70, 50)
upper1 = (10, 255, 255)
lower2 = (170, 70, 50)
upper2 = (180, 255, 255)
```

The binary mask is:

$
M(u,v) = \begin{cases}
1 & \text{if } \text{HSV}(u,v) \in \text{red-range}, \
0 & \text{otherwise}
\end{cases}
$

To clean noise:

* **GaussianBlur** (remove pixel-level noise)
* **morphological opening** (remove specks)
* **closing** (fill small holes)

##### **Theory: Polygon Approximation**
Now that we have a cleaned mask, we try to identify contours and approximate their shape with polygons. 
1. Contour Detection (`cv2.findContours`):

    Contours are simply continuous curves that follow the boundary of a blob in a binary image. Given our cleaned red mask, `cv2.findContours()` scans the image using border-following algorithms (like Suzuki-Abe) to find the set of points that lie on the edge of a red object. We use `cv2.RETR_EXTERNAL` so that we only detect the outermost contour, ignoring holes and nested shapes. We use `cv2.CHAIN_APPROX_NONE` to keep as much detail as possible. Although this increases computation, this is done once at the start and so extra computation doesn't negatively impact main loop frequency. 

2. Polygon Approximation (Ramer–Douglas–Peucker Algorithm):

    Since contours can contain hundreds or thousands of points (inefficient and unnecessary for defining obstacle in the world frame), we simplify them using:

    `poly = cv2.approxPolyDP(cnt, epsilon, True)`
    
    which uses the uses the Ramer–Douglas–Peucker (RDP) algorithm which converts a dense contour into a simpler polygon while preserving overall shape (subject to an epsilon defining the allowed distance between the polygon and a point in the contour that is not in the polygon). We define epsilon proportional to the contour perimiter to account for different shape sizes. We use a small epsilon (0.1% of the perimeter) to keep as much detail as possible. As mentioned above, this increased computation does not negatively impact the main loop frequency as this is done once at the start and then the obstacle information is stored in the grid which is a fixed size.

### **1.5 convertPolygonsToWorld**

Each polygon vertex (camera pixel) is converted via:

$
\mathbf{p}_{world} = H * \mathbf{p}_{camera}
$

This produces **world-frame polygons**, used by global navigation.



### **1.6 findGoalPos(frame)**

Detects ArUco marker with ID **3** and converts its centre from camera to world using the same homography.
This keeps all environment features in a consistent coordinate system.


### **1.7 createGrid**

##### **Purpose**

Create an occupancy grid representation of the environment.

We use the arena’s global size (from ArUco detection) to define grid bounds.
Each obstacle polygon is rasterised into the grid and stored with a value of -1, representing untraversable space.

Grid resolution is chosen as a compromise between:

* allowing global planning to identify a smooth and accurate path
* planning runtime

We chose to use 200 cells per meter.

---

# **2. Visualisation Utilities**

We modularise the visualisation system so individual components can be enabled/disabled independently.
These functions operate solely in **camera pixel space**, using the detected pixels or the inverse homography when visualising global-calculted points (like the global path).

### **2.1 visualiseArena**

Draws the arena bounding box.

### **2.2 visualiseRobotPose**

Draws the marker bounding box, robot’s center position and orientation as an arrow.

### **2.3 visualiseObstacles**

Draws each obstacle polygon.

### **2.4 visualiseGoalPos**

Draws a bounding box around the goal marker and highlight's the center position.

### **2.5 visualiseGlobalPoint / visualiseGlobalPath**
Given a world coordinate ($\mathbf{p}_{world}$), convert it to camera pixel coordinates using the inverse of the homography:

$
\mathbf{p}_{camera} = H^{-1} \mathbf{p}_{world}
$

Then draw that pixel location on the camera frame. To visualise a path, we also draw straight lines between sequential points.

The path visualisation includes a sampling safeguard:

```python
if len(path) > 200:
    path = sample(path, 200)
```

This prevents the visualiser from slowing the main loop (and disrupting the operational frequency) during long episodes.

---

# **3. Live Feedback**
Since we have defined our detection functions in a modular way (they take a frame and detect the specific feature), we are able to obtain the live robot pose by obtaining a frame from the initialised video capture inside the main loop and calling `getRobotPose(frame)`. 

---
# **4. Example usage**

In [ ]:
visionInstanceDemo = Vision() # Calls getEnvrionment which detects and stores all static environmental features

# Obtain live robot pose visualisation
if not visionInstanceDemo.cap.isOpened(): # Use the already initialised videoCapture 
    print("Error: Could not access the webcam.")
    exit()

while True:
    ret, frame = visionInstanceDemo.cap.read()
    if not ret:
        print("Camera failed to capture the frame.")
        break

    vis = frame.copy() # Create a copy of the frame to be able to visualise the image without affecting detection

    # Redraw arena outline
    visionInstanceDemo.visualiseArena(vis, visionInstanceDemo.arena_corners_pixels)

    # Detect and draw robot pose
    position, robot_heading_angle = visionInstanceDemo.getRobotPoseAndVisualise(frame, vis)
    if (position is None or robot_heading_angle is None):
        continue
    [X, Y] = position

    path = [[X*100, Y*100], [40, 40], [50, 60], [70, 70]] # Visualise a random global path
    visionInstanceDemo.visualiseGlobalPath(vis, path)

    # Show the live window
    cv2.imshow("Live Robot Pose", vis)

    # Exit on Q
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

visionInstanceDemo.cap.release()
cv2.destroyAllWindows()

---
---

# **Global Navigation**

This part explains the global navigation module (`global_nav.py`) used for path planning in the Mobile Robotics Project.

The global navigation system finds a collision-free path from a **start** position to a **goal** position on a 2D occupancy grid. It uses:

1. **Obstacle Expansion** - Grow obstacles by the robot's size so we can treat the robot as a point  
2. **A* or Dijkstra Search** - Find the shortest path on the expanded grid  
3. **Path Simplification** - Reduce the cell-by-cell path to key waypoints  
4. **Visualization** - Display the results on the map

---

### 1. Occupancy Grid

The arena is represented as a **200x200 cell grid** (defined in `utils.py`):

| Value | Meaning |
|-------|---------|
| `0`   | Free space |
| `-1`  | Obstacle |

#### Coordinate Systems

There are two coordinate systems:

1. **Grid coordinates** `(row, col)`:
   - Row 0 = TOP, row 199 = BOTTOM  
   - Column 0 = LEFT, column 199 = RIGHT

2. **Real-world coordinates** `(x_cm, y_cm)`:
   - Origin (0, 0) = BOTTOM-LEFT  
   - X increases to the right, Y increases upward

Conversion functions in `utils.py`:
- `real_to_grid(coord)`
- `grid_to_real(coord)`
- `cm_to_cell(cm_value)`
- `cell_to_cm(cell_index)`

---

### 2. Obstacle Expansion

This function grows obstacles outward by the robot's size so that the pathfinding algorithm can treat the robot as a single point instead of checking its full footprint at every step.

The kernel size determines how much we expand the obstacle in all directions. We expand by half of the robot's size and add a small safety margin to make sure no overlap will occur between the obstacle and the robot.

---

### 3. Pathfinding Algorithm

The user can choose between 2 pathfinding algorithms. Both work using the 8 connected neighbouring cells on the grid.
#### Djikstra

#### A star

**Heuristic Function**

- A* uses a heuristic to estimate distance to the goal. Because movement is 8-directional, we use the **octile distance**. 
(Reference for heuristic function choice : https://theory.stanford.edu/~amitp/GameProgramming/Heuristics.html)

- Diagonal moves cost `sqrt(2) = ~1.414`, straight moves cost `1`.  Optimal path: move diagonally as much as possible, then straight.

**Pathfinding**

A star works in the same way as Dijkstra's Algorithm but adds the heuristic function to guide the path finding towards the goal and therefore explore less cells.

---

### 4. Path Simplification

The path returned by the pathfinding algorithms give each cell the robot passes on. This is not efficient at all so we simplify this path by only taking the **corner waypoints**. 

The code looks at each cell in the path and checks the direction, if the direction of point 1 is the same as point 2, we remove point 2 from the list of cells. Then so on until we reach a point with a different direction, this one will be kept it in the simplified path list.


---
---

# **Local Navigation**

---
---

# **Filtering**

---
---

# **Motion Control**